# Consolidated Results Summary and Mid-Sem Evidence Pack

This notebook consolidates the main experimental outputs generated so far for the dissertation. It does not introduce new modelling experiments. The purpose is to collect the key results, selected figures, artefacts, findings, and limitations into a report-ready evidence pack for the mid-semester dissertation review.

## 1. Setup and Paths

The consolidation notebook uses saved outputs from the earlier experiment notebooks. No new model training is performed in this step.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai'
else:
    BASE = '..'

METRICS_DIR = f'{BASE}/reports/metrics'
FIG_DIR     = f'{BASE}/reports/figures'
EXPLAIN_DIR = f'{BASE}/reports/explainability'
ROBUST_DIR  = f'{BASE}/reports/robustness'
SUMMARY_DIR = f'{BASE}/reports/summary'

os.makedirs(SUMMARY_DIR, exist_ok=True)
print('Paths set up.')


Mounted at /content/drive
Paths set up.


## 2. Load Result Tables

In [2]:
classical_results     = pd.read_csv(f'{METRICS_DIR}/classical_baseline_results_fd001.csv')
deep_results          = pd.read_csv(f'{METRICS_DIR}/deep_learning_baseline_results_fd001.csv')
window_aligned        = pd.read_csv(f'{METRICS_DIR}/classical_vs_deep_window_aligned_comparison_fd001.csv')
multiview_results     = pd.read_csv(f'{METRICS_DIR}/multiview_vs_baselines_comparison_fd001.csv')
robustness_summary    = pd.read_csv(f'{ROBUST_DIR}/robustness_summary_multiview_fd001.csv')
rul_range_summary     = pd.read_csv(f'{ROBUST_DIR}/rul_range_error_analysis_multiview_fd001.csv')
bias_summary          = pd.read_csv(f'{ROBUST_DIR}/prediction_bias_summary_multiview_fd001.csv')
shap_ranking          = pd.read_csv(f'{EXPLAIN_DIR}/xgboost_shap_feature_ranking_fd001.csv')
view_masking          = pd.read_csv(f'{EXPLAIN_DIR}/multiview_view_masking_explainability_fd001.csv')

print('classical_results:  ', classical_results.shape)
print('deep_results:       ', deep_results.shape)
print('window_aligned:     ', window_aligned.shape)
print('multiview_results:  ', multiview_results.shape)
print('robustness_summary: ', robustness_summary.shape)


classical_results:   (15, 5)
deep_results:        (2, 6)
window_aligned:      (3, 7)
multiview_results:   (5, 6)
robustness_summary:  (21, 13)


## 3. Consolidated Model Comparison

The consolidated comparison uses the window-aligned validation basis so that sequence-based models and the classical baseline are compared on the same validation rows. MultiViewGRUFusion achieved the lowest validation RMSE among the evaluated models, followed by XGBoost using engineered degradation features.

In [3]:
model_summary = multiview_results.copy()
model_summary = model_summary.rename(columns={'feature_or_views': 'input_representation'})

keep_cols = ['model_group', 'model', 'input_representation',
             'validation_rmse', 'validation_mae', 'validation_r2']
model_summary = model_summary[keep_cols].sort_values('validation_rmse').reset_index(drop=True)

model_summary.to_csv(f'{SUMMARY_DIR}/consolidated_model_comparison_fd001.csv', index=False)
print(model_summary.to_string(index=False))


             model_group              model                            input_representation  validation_rmse  validation_mae  validation_r2
Multi-view Deep Learning MultiViewGRUFusion sensor_sequence_view + derived_degradation_view          12.0657          8.9406         0.9168
               Classical            XGBoost                                               C          12.4894          9.2675         0.9109
Multi-view Deep Learning     DerivedOnlyMLP                        derived_degradation_view          13.1451          9.4204         0.9013
           Deep Learning                GRU                                   B (window=30)          13.1605          9.7182         0.9010
           Deep Learning              CNN1D                                   B (window=30)          18.1509         14.0382         0.8118


## 4. Best Model Summary

The current best model is selected based on validation RMSE on the window-aligned validation set. This is not claimed as a final result; it is the current best result at the mid-semester stage.

In [4]:
best = model_summary.iloc[0].to_dict()

best_model_summary = pd.DataFrame([{
    'current_best_model':    best['model'],
    'model_group':           best['model_group'],
    'input_representation':  best['input_representation'],
    'validation_rmse':       best['validation_rmse'],
    'validation_mae':        best['validation_mae'],
    'validation_r2':         best['validation_r2'],
    'comparison_basis':      'window-aligned validation rows',
    'status':                'current best validation result'
}])

best_model_summary.to_csv(f'{SUMMARY_DIR}/best_model_summary_fd001.csv', index=False)
print(best_model_summary.to_string(index=False))


current_best_model              model_group                            input_representation  validation_rmse  validation_mae  validation_r2               comparison_basis                         status
MultiViewGRUFusion Multi-view Deep Learning sensor_sequence_view + derived_degradation_view          12.0657          8.9406         0.9168 window-aligned validation rows current best validation result


## 5. Step-wise Progress Summary

In [5]:
step_summary = pd.DataFrame([
    {'step': 'Step 1', 'title': 'Dataset Understanding and Problem Formulation',
     'main_output': 'FD001 loaded, RUL regression formulated, leakage risks documented'},
    {'step': 'Step 2', 'title': 'EDA and Feature Selection',
     'main_output': 'Variable sensors identified, feature decision table and EDA plots generated'},
    {'step': 'Step 3', 'title': 'Baseline-ready Preprocessing',
     'main_output': 'RUL capping, engine-level split, scaling, Feature Sets A/B/C created'},
    {'step': 'Step 4', 'title': 'Classical Baselines',
     'main_output': 'XGBoost/C established as strongest classical baseline (RMSE 12.49, window-aligned)'},
    {'step': 'Step 5', 'title': 'Deep Learning Baselines',
     'main_output': 'CNN1D and GRU trained; GRU became best deep baseline (RMSE 13.16, window-aligned)'},
    {'step': 'Step 6', 'title': 'Initial Multi-view Model',
     'main_output': 'MultiViewGRUFusion trained; achieved RMSE 12.07 on window-aligned validation'},
    {'step': 'Step 7', 'title': 'Initial Explainability',
     'main_output': 'Feature importance, SHAP, view masking, and local case analysis completed'},
    {'step': 'Step 8', 'title': 'Initial Robustness and Trustworthiness',
     'main_output': 'Noise, masking, RUL-range, bias, stability, and MC Dropout checks completed'},
])

step_summary.to_csv(f'{SUMMARY_DIR}/stepwise_progress_summary_fd001.csv', index=False)
print(step_summary.to_string(index=False))


  step                                         title                                                                        main_output
Step 1 Dataset Understanding and Problem Formulation                  FD001 loaded, RUL regression formulated, leakage risks documented
Step 2                     EDA and Feature Selection        Variable sensors identified, feature decision table and EDA plots generated
Step 3                  Baseline-ready Preprocessing               RUL capping, engine-level split, scaling, Feature Sets A/B/C created
Step 4                           Classical Baselines XGBoost/C established as strongest classical baseline (RMSE 12.49, window-aligned)
Step 5                       Deep Learning Baselines  CNN1D and GRU trained; GRU became best deep baseline (RMSE 13.16, window-aligned)
Step 6                      Initial Multi-view Model       MultiViewGRUFusion trained; achieved RMSE 12.07 on window-aligned validation
Step 7                        Initial Explainabi

## 6. Dissertation Objective Coverage

In [6]:
objective_coverage = pd.DataFrame([
    {'dissertation_component': 'Predictive maintenance',
     'evidence_generated': 'RUL prediction on NASA C-MAPSS FD001 using engine degradation trajectories',
     'status': 'Covered'},
    {'dissertation_component': 'Deep learning',
     'evidence_generated': 'CNN1D, GRU, DerivedOnlyMLP, and MultiViewGRUFusion models trained',
     'status': 'Covered'},
    {'dissertation_component': 'Multimodal / multi-view learning',
     'evidence_generated': 'Sensor sequence view and derived degradation feature view fused in MultiViewGRUFusion',
     'status': 'Covered as benchmark-driven multi-view learning'},
    {'dissertation_component': 'Explainability',
     'evidence_generated': 'XGBoost feature importance, SHAP analysis, view masking, and local case analysis',
     'status': 'Initial coverage complete'},
    {'dissertation_component': 'Trustworthiness',
     'evidence_generated': 'Noise robustness, random masking, missing-view, RUL-range, bias, stability, and MC Dropout checks',
     'status': 'Initial model-level coverage complete'},
    {'dissertation_component': 'Industrial assets',
     'evidence_generated': 'C-MAPSS turbofan engine degradation benchmark used as industrial asset proxy',
     'status': 'Covered within benchmark scope'},
])

objective_coverage.to_csv(f'{SUMMARY_DIR}/dissertation_objective_coverage_fd001.csv', index=False)
print(objective_coverage.to_string(index=False))


          dissertation_component                                                                                evidence_generated                                          status
          Predictive maintenance                        RUL prediction on NASA C-MAPSS FD001 using engine degradation trajectories                                         Covered
                   Deep learning                                 CNN1D, GRU, DerivedOnlyMLP, and MultiViewGRUFusion models trained                                         Covered
Multimodal / multi-view learning             Sensor sequence view and derived degradation feature view fused in MultiViewGRUFusion Covered as benchmark-driven multi-view learning
                  Explainability                  XGBoost feature importance, SHAP analysis, view masking, and local case analysis                       Initial coverage complete
                 Trustworthiness Noise robustness, random masking, missing-view, RUL-range, bias, stabili

## 7. Explainability Summary

The explainability results are treated as model-level explanations. They do not imply physical causality.

In [7]:
top_shap = shap_ranking.head(5).copy()
top_shap.to_csv(f'{SUMMARY_DIR}/top_shap_features_fd001.csv', index=False)
print('Top 5 SHAP features:')
print(top_shap.to_string(index=False))


Top 5 SHAP features:
                    feature  mean_abs_shap
 sensor_measurement_4_rmean       6.651353
sensor_measurement_11_rmean       5.529419
                cycle_index       5.352144
sensor_measurement_15_rmean       3.879201
sensor_measurement_11_delta       3.026035


In [8]:
explainability_summary = pd.DataFrame([
    {'explainability_check': 'XGBoost native feature importance',
     'main_finding': 'Rolling mean degradation features dominate model importance'},
    {'explainability_check': 'SHAP global summary',
     'main_finding': 'sensor_measurement_4_rmean, sensor_measurement_11_rmean, and cycle_index are top contributors'},
    {'explainability_check': 'SHAP dependence plots',
     'main_finding': 'Top features show directional contribution patterns to predicted RUL'},
    {'explainability_check': 'View masking',
     'main_finding': 'Derived degradation view is the dominant driver; sensor sequence view adds complementary signal'},
    {'explainability_check': 'Local case analysis',
     'main_finding': 'Representative good, under-predicted, and over-predicted cases were inspected, including a mid-range good prediction'},
])

explainability_summary.to_csv(f'{SUMMARY_DIR}/explainability_summary_fd001.csv', index=False)
print(explainability_summary.to_string(index=False))


             explainability_check                                                                                                         main_finding
XGBoost native feature importance                                                          Rolling mean degradation features dominate model importance
              SHAP global summary                        sensor_measurement_4_rmean, sensor_measurement_11_rmean, and cycle_index are top contributors
            SHAP dependence plots                                                 Top features show directional contribution patterns to predicted RUL
                     View masking                      Derived degradation view is the dominant driver; sensor sequence view adds complementary signal
              Local case analysis Representative good, under-predicted, and over-predicted cases were inspected, including a mid-range good prediction


## 8. Robustness and Trustworthiness Summary

These robustness checks provide initial model-level trustworthiness evidence. They do not prove production reliability or operational readiness.

In [9]:
selected_conditions = [
    'baseline',
    'sensor_sequence_noise_0.2',
    'derived_feature_noise_0.2',
    'combined_noise_0.2',
    'sensor_sequence_random_mask_0.3',
    'derived_feature_random_mask_0.3',
    'combined_random_mask_0.3',
    'sensor_sequence_masked',
    'derived_view_masked',
]

robustness_report_table = robustness_summary[
    robustness_summary['condition'].isin(selected_conditions)
][['condition', 'perturbation_type', 'rmse', 'mae', 'r2', 'rmse_increase_pct']].sort_values('rmse').copy()

robustness_report_table.to_csv(f'{SUMMARY_DIR}/robustness_report_table_fd001.csv', index=False)
print(robustness_report_table.to_string(index=False))


                      condition              perturbation_type    rmse     mae      r2  rmse_increase_pct
                       baseline                           none 12.0657  8.9406  0.9168               0.00
      sensor_sequence_noise_0.2          sensor_sequence_noise 12.1026  8.9582  0.9163               0.31
      derived_feature_noise_0.2          derived_feature_noise 12.5102  9.2110  0.9106               3.68
sensor_sequence_random_mask_0.3 sensor_sequence_random_masking 12.5428  9.7108  0.9101               3.95
             combined_noise_0.2                 combined_noise 12.5906  9.1948  0.9094               4.35
         sensor_sequence_masked                   view_masking 15.5505 12.6154  0.8618              28.88
derived_feature_random_mask_0.3 derived_feature_random_masking 17.7557 14.4966  0.8199              47.16
       combined_random_mask_0.3        combined_random_masking 19.0455 15.8509  0.7928              57.85
            derived_view_masked               

In [10]:
trustworthiness_summary = pd.DataFrame([
    {'check': 'Sensor sequence noise',
     'main_finding': 'Model performance remained stable under sensor sequence noise (RMSE +0.31% at noise_std=0.20)'},
    {'check': 'Derived feature perturbation',
     'main_finding': 'Model was more sensitive to derived feature perturbation than sensor sequence perturbation'},
    {'check': 'Random masking',
     'main_finding': 'Derived feature masking caused larger degradation than sensor sequence masking'},
    {'check': 'Missing view',
     'main_finding': 'Full derived-view masking caused severe performance collapse (RMSE 42.57, R2 -0.04)'},
    {'check': 'RUL range-wise error',
     'main_finding': 'Near-failure predictions most accurate (RMSE 4.43); early/capped range showed higher error and under-prediction'},
    {'check': 'Prediction bias',
     'main_finding': 'Model showed slight overall under-prediction tendency; 90.5% of predictions within 20 cycles'},
    {'check': 'Repeated perturbation stability',
     'main_finding': 'Low RMSE variation across random seeds (std 0.009–0.034 at noise_std=0.1)'},
    {'check': 'MC Dropout uncertainty',
     'main_finding': 'Prediction uncertainty showed moderate alignment with absolute error (correlation 0.49)'},
])

trustworthiness_summary.to_csv(f'{SUMMARY_DIR}/trustworthiness_summary_fd001.csv', index=False)
print(trustworthiness_summary.to_string(index=False))


                          check                                                                                                    main_finding
          Sensor sequence noise                   Model performance remained stable under sensor sequence noise (RMSE +0.31% at noise_std=0.20)
   Derived feature perturbation                      Model was more sensitive to derived feature perturbation than sensor sequence perturbation
                 Random masking                                  Derived feature masking caused larger degradation than sensor sequence masking
                   Missing view                             Full derived-view masking caused severe performance collapse (RMSE 42.57, R2 -0.04)
           RUL range-wise error Near-failure predictions most accurate (RMSE 4.43); early/capped range showed higher error and under-prediction
                Prediction bias                    Model showed slight overall under-prediction tendency; 90.5% of predictions within 20

## 9. Selected Figures for Mid-Sem Report

Only a subset of generated figures is recommended for inclusion in the mid-sem report. The filenames below match the generated artefacts on disk.

In [11]:
selected_figures = pd.DataFrame([
    {'figure_file': 'feature_correlation_ranking_fd001.png',
     'report_section': 'EDA', 'why_selected': 'Shows sensor association with RUL'},
    {'figure_file': 'sensor_trends_selected_fd001.png',
     'report_section': 'EDA', 'why_selected': 'Shows degradation trends over engine lifecycle'},
    {'figure_file': 'classical_vs_deep_rmse_comparison_fd001.png',
     'report_section': 'Baseline comparison', 'why_selected': 'Compares classical and deep learning baselines on window-aligned set'},
    {'figure_file': 'multiview_vs_baselines_rmse_fd001.png',
     'report_section': 'Multi-view model results', 'why_selected': 'Shows MultiViewGRUFusion compared with all baselines'},
    {'figure_file': 'actual_vs_predicted_multiview_fd001.png',
     'report_section': 'Multi-view model results', 'why_selected': 'Shows prediction quality of the best model'},
    {'figure_file': 'shap_summary_xgboost_fd001.png',
     'report_section': 'Explainability', 'why_selected': 'Shows global SHAP explanation for XGBoost baseline'},
    {'figure_file': 'multiview_view_masking_rmse_fd001.png',
     'report_section': 'Explainability', 'why_selected': 'Shows relative importance of input views via masking'},
    {'figure_file': 'robustness_summary_rmse_multiview_fd001.png',
     'report_section': 'Robustness', 'why_selected': 'Summarizes noise and masking robustness across conditions'},
    {'figure_file': 'rul_range_error_multiview_fd001.png',
     'report_section': 'Robustness', 'why_selected': 'Shows lifecycle-region-wise prediction error'},
    {'figure_file': 'uncertainty_vs_error_multiview_fd001.png',
     'report_section': 'Trustworthiness', 'why_selected': 'Shows MC Dropout uncertainty relation with absolute error'},
])

# Verify all figure files exist
missing = [r['figure_file'] for _, r in selected_figures.iterrows()
           if not os.path.exists(f'{FIG_DIR}/{r["figure_file"]}')]
if missing:
    print(f'WARNING: missing figures: {missing}')
else:
    print('All selected figures verified on disk.')

selected_figures.to_csv(f'{SUMMARY_DIR}/selected_midsem_figures_fd001.csv', index=False)
print(selected_figures.to_string(index=False))


All selected figures verified on disk.
                                figure_file           report_section                                                         why_selected
      feature_correlation_ranking_fd001.png                      EDA                                    Shows sensor association with RUL
           sensor_trends_selected_fd001.png                      EDA                       Shows degradation trends over engine lifecycle
classical_vs_deep_rmse_comparison_fd001.png      Baseline comparison Compares classical and deep learning baselines on window-aligned set
      multiview_vs_baselines_rmse_fd001.png Multi-view model results                 Shows MultiViewGRUFusion compared with all baselines
    actual_vs_predicted_multiview_fd001.png Multi-view model results                           Shows prediction quality of the best model
             shap_summary_xgboost_fd001.png           Explainability                   Shows global SHAP explanation for XGBoost base

## 10. Key Findings

In [12]:
key_findings = pd.DataFrame([
    {'finding_no': 1, 'finding': 'Derived degradation features substantially improved RUL prediction compared with raw sensor features.'},
    {'finding_no': 2, 'finding': 'GRU performed better than CNN1D for sequence-based RUL prediction on selected sensor windows.'},
    {'finding_no': 3, 'finding': 'The initial MultiViewGRUFusion model achieved the best validation result by combining sensor sequence and derived degradation views.'},
    {'finding_no': 4, 'finding': 'Explainability analysis showed that rolling mean degradation features were dominant contributors in the XGBoost baseline.'},
    {'finding_no': 5, 'finding': 'View masking showed that the derived degradation view was the dominant contributor in the fusion model, while the sensor sequence view added complementary signal.'},
    {'finding_no': 6, 'finding': 'Robustness checks showed stronger sensitivity to derived feature perturbation than to raw sensor sequence perturbation.'},
    {'finding_no': 7, 'finding': 'Near-failure RUL predictions were more accurate than early/capped-region predictions.'},
    {'finding_no': 8, 'finding': 'MC Dropout uncertainty showed moderate alignment with absolute prediction error, providing an initial uncertainty signal.'},
])

key_findings.to_csv(f'{SUMMARY_DIR}/key_findings_fd001.csv', index=False)
print(key_findings.to_string(index=False))


 finding_no                                                                                                                                                            finding
          1                                                              Derived degradation features substantially improved RUL prediction compared with raw sensor features.
          2                                                                      GRU performed better than CNN1D for sequence-based RUL prediction on selected sensor windows.
          3                               The initial MultiViewGRUFusion model achieved the best validation result by combining sensor sequence and derived degradation views.
          4                                          Explainability analysis showed that rolling mean degradation features were dominant contributors in the XGBoost baseline.
          5 View masking showed that the derived degradation view was the dominant contributor in the fusion model, while the

## 11. Current Limitations

In [13]:
limitations = pd.DataFrame([
    {'limitation': 'Experiments are currently limited to C-MAPSS FD001.',
     'mitigation_or_next_step': 'Extend to FD002/FD004 or other datasets in final phase if time permits.'},
    {'limitation': 'Multimodality is operationalized as benchmark-driven multi-view learning, not full native multimodality with image/text/audio.',
     'mitigation_or_next_step': 'Clearly document this scope boundary in the report.'},
    {'limitation': 'Results are based on one validation split.',
     'mitigation_or_next_step': 'Repeated runs or additional split validation can be considered in final phase.'},
    {'limitation': 'Neural-network explainability is limited to view masking at this stage.',
     'mitigation_or_next_step': 'Integrated Gradients or gradient-based explanations may be considered later.'},
    {'limitation': 'MC Dropout uncertainty is not calibrated.',
     'mitigation_or_next_step': 'Treat as an initial uncertainty signal only.'},
    {'limitation': 'Production deployment and real-time integration are outside current scope.',
     'mitigation_or_next_step': 'Position work as benchmark-based applied research prototype.'},
])

limitations.to_csv(f'{SUMMARY_DIR}/limitations_and_next_steps_fd001.csv', index=False)
print(limitations.to_string(index=False))


                                                                                                                    limitation                                                        mitigation_or_next_step
                                                                           Experiments are currently limited to C-MAPSS FD001.        Extend to FD002/FD004 or other datasets in final phase if time permits.
Multimodality is operationalized as benchmark-driven multi-view learning, not full native multimodality with image/text/audio.                            Clearly document this scope boundary in the report.
                                                                                    Results are based on one validation split. Repeated runs or additional split validation can be considered in final phase.
                                                       Neural-network explainability is limited to view masking at this stage.   Integrated Gradients or gradient-based explanat

## 12. Report-ready Narrative Summary

The dissertation work has progressed from dataset understanding to model development, explainability, and robustness checks. FD001 from the NASA C-MAPSS dataset was used as the initial benchmark subset. The prediction task was formulated as capped RUL regression. After EDA and preprocessing, three feature sets were prepared and evaluated using classical models. XGBoost with engineered degradation features provided a strong classical baseline.

Sequence-based deep learning baselines were then trained using 30-cycle windows of selected sensor measurements. GRU performed better than CNN1D but did not exceed the engineered-feature XGBoost baseline. An initial multi-view deep learning model was then developed by combining a raw sensor sequence view with a derived degradation feature view. The MultiViewGRUFusion model achieved the best validation RMSE among the evaluated models on the window-aligned validation set.

Initial explainability analysis was performed using XGBoost feature importance, SHAP analysis, view masking, and local case analysis. The results showed that rolling mean degradation features were dominant contributors and that the derived degradation view was the primary driver of the fusion model. Robustness checks showed that the model was stable under small sensor-sequence perturbations but more sensitive to derived-feature perturbations and masking. RUL range-wise analysis showed better performance near failure than in the early/capped RUL region. MC Dropout provided an initial uncertainty signal with moderate alignment to actual prediction error.

Overall, the current work provides a benchmark-based applied implementation of explainable and trustworthy multi-view deep learning for predictive maintenance, within the scope of FD001 RUL prediction.

## 13. Mid-Sem Report Asset Checklist

In [14]:
midsem_asset_checklist = pd.DataFrame([
    {'asset': 'Dataset description',          'status': 'Ready', 'source': 'Step 1'},
    {'asset': 'EDA summary',                  'status': 'Ready', 'source': 'Step 2'},
    {'asset': 'Preprocessing strategy',       'status': 'Ready', 'source': 'Step 3'},
    {'asset': 'Classical baseline results',   'status': 'Ready', 'source': 'Step 4'},
    {'asset': 'Deep learning baseline results','status': 'Ready', 'source': 'Step 5'},
    {'asset': 'Multi-view model results',      'status': 'Ready', 'source': 'Step 6'},
    {'asset': 'Explainability outputs',        'status': 'Ready', 'source': 'Step 7'},
    {'asset': 'Robustness outputs',            'status': 'Ready', 'source': 'Step 8'},
    {'asset': 'Consolidated model comparison', 'status': 'Ready', 'source': 'Step 9'},
    {'asset': 'Selected figures for report',   'status': 'Ready', 'source': 'Step 9'},
    {'asset': 'Limitations and future work',   'status': 'Ready', 'source': 'Step 9'},
])

midsem_asset_checklist.to_csv(f'{SUMMARY_DIR}/midsem_asset_checklist_fd001.csv', index=False)
print(midsem_asset_checklist.to_string(index=False))


                         asset status source
           Dataset description  Ready Step 1
                   EDA summary  Ready Step 2
        Preprocessing strategy  Ready Step 3
    Classical baseline results  Ready Step 4
Deep learning baseline results  Ready Step 5
      Multi-view model results  Ready Step 6
        Explainability outputs  Ready Step 7
            Robustness outputs  Ready Step 8
 Consolidated model comparison  Ready Step 9
   Selected figures for report  Ready Step 9
   Limitations and future work  Ready Step 9


## 14. Generated Artefacts

In [15]:
print('Generated summary files:')
for f in sorted(os.listdir(SUMMARY_DIR)):
    print(f'  {f}')


Generated summary files:
  best_model_summary_fd001.csv
  consolidated_model_comparison_fd001.csv
  dissertation_objective_coverage_fd001.csv
  explainability_summary_fd001.csv
  key_findings_fd001.csv
  limitations_and_next_steps_fd001.csv
  midsem_asset_checklist_fd001.csv
  robustness_report_table_fd001.csv
  selected_midsem_figures_fd001.csv
  stepwise_progress_summary_fd001.csv
  top_shap_features_fd001.csv
  trustworthiness_summary_fd001.csv
